In [ ]:
from interleaving_algorithm import rank_based_interleaving, analyze_interleaving_results
import pandas as pd
import sys

In [ ]:
def diagnose_query(df_A, df_B, qid, easy_negatives, balance_threshold=2):
    """
    详细诊断单个查询的交错过程
    
    Args:
        df_A: 系统A的排名结果
        df_B: 系统B的排名结果
        qid: 要诊断的查询ID
        easy_negatives: 负样本
        balance_threshold: 平衡阈值
    """
    print("=" * 100)
    print(f"🔍 诊断查询: {qid}")
    print("=" * 100)
    
    # 1. 筛选该查询的数据
    df_A_query = df_A[df_A['qid'] == qid].copy()
    df_B_query = df_B[df_B['qid'] == qid].copy()
    
    # 按score排序
    df_A_query = df_A_query.sort_values('score', ascending=False).reset_index(drop=True)
    df_B_query = df_B_query.sort_values('score', ascending=False).reset_index(drop=True)
    
    print(f"\n📊 输入数据统计:")
    print(f"  系统A文档数: {len(df_A_query)}")
    print(f"  系统B文档数: {len(df_B_query)}")
    
    # 2. 显示两个系统的排名
    print(f"\n📋 系统A的排名（按score降序）:")
    print(f"  {'排名':<6} {'docno':<20} {'score':<10}")
    print(f"  {'-'*40}")
    for idx, row in df_A_query.head(15).iterrows():
        print(f"  {idx:<6} {row['docno']:<20} {row['score']:<10.2f}")
    if len(df_A_query) > 15:
        print(f"  ... 还有 {len(df_A_query)-15} 个文档")
    
    print(f"\n📋 系统B的排名（按score降序）:")
    print(f"  {'排名':<6} {'docno':<20} {'score':<10}")
    print(f"  {'-'*40}")
    for idx, row in df_B_query.head(15).iterrows():
        print(f"  {idx:<6} {row['docno']:<20} {row['score']:<10.2f}")
    if len(df_B_query) > 15:
        print(f"  ... 还有 {len(df_B_query)-15} 个文档")
    
    # 3. 找出Both文档
    docs_A = set(df_A_query['docno'].tolist())
    docs_B = set(df_B_query['docno'].tolist())
    both_docs = docs_A & docs_B
    
    print(f"\n🔗 重叠分析:")
    print(f"  系统A独有: {len(docs_A - docs_B)} 个文档")
    print(f"  系统B独有: {len(docs_B - docs_A)} 个文档")
    print(f"  两者共有: {len(both_docs)} 个文档")
    
    if both_docs:
        print(f"\n  共有文档列表:")
        for doc in sorted(both_docs):
            rank_A = df_A_query[df_A_query['docno'] == doc].index[0]
            rank_B = df_B_query[df_B_query['docno'] == doc].index[0]
            score_A = df_A_query[df_A_query['docno'] == doc]['score'].values[0]
            score_B = df_B_query[df_B_query['docno'] == doc]['score'].values[0]
            print(f"    {doc:<20} A排名:{rank_A:<3} (score:{score_A:>6.2f})  "
                  f"B排名:{rank_B:<3} (score:{score_B:>6.2f})")
    
    # 4. 执行交错并跟踪过程
    print(f"\n🔄 交错扫描过程（balance_threshold={balance_threshold}）:")
    print(f"  {'k':<4} {'A[k]':<20} {'B[k]':<20} {'动作':<30} {'当前A/B计数':<15}")
    print(f"  {'-'*95}")
    
    # 手动模拟扫描过程以展示细节
    seen_docs = set()
    selections = []
    count_a_only = 0
    count_b_only = 0
    count_both = 0
    
    k = 0
    max_k = min(20, max(len(df_A_query), len(df_B_query)))
    
    while len(selections) < 9 and k < max_k:
        doc_A = df_A_query.iloc[k]['docno'] if k < len(df_A_query) else None
        doc_B = df_B_query.iloc[k]['docno'] if k < len(df_B_query) else None
        
        action = ""
        
        # 情况1: Both
        if doc_A is not None and doc_B is not None and doc_A == doc_B:
            if doc_A not in seen_docs:
                selections.append(('Both', doc_A))
                seen_docs.add(doc_A)
                count_both += 1
                action = f"✓ Both: {doc_A}"
            else:
                action = f"⊗ 已存在: {doc_A}"
        
        # 情况2: 不同
        else:
            balance_diff_a = count_a_only - count_b_only
            balance_diff_b = count_b_only - count_a_only
            
            # 处理A
            if doc_A is not None and doc_A not in seen_docs:
                if balance_diff_a >= balance_threshold:
                    action += f"⊗ 跳过A:{doc_A[:15]} (A太多) "
                else:
                    selections.append(('A-Only', doc_A))
                    seen_docs.add(doc_A)
                    count_a_only += 1
                    action += f"✓ A:{doc_A[:15]} "
                    
                    if len(selections) == 9:
                        action += "(已满9个)"
                        print(f"  {k:<4} {str(doc_A):<20} {str(doc_B):<20} {action:<30} A:{count_a_only} B:{count_b_only}")
                        break
            
            # 处理B
            if doc_B is not None and doc_B not in seen_docs:
                if balance_diff_b >= balance_threshold:
                    action += f"⊗ 跳过B:{doc_B[:15]} (B太多)"
                else:
                    selections.append(('B-Only', doc_B))
                    seen_docs.add(doc_B)
                    count_b_only += 1
                    action += f"✓ B:{doc_B[:15]}"
        
        count_str = f"A:{count_a_only} B:{count_b_only}"
        print(f"  {k:<4} {str(doc_A)[:20]:<20} {str(doc_B)[:20]:<20} {action:<30} {count_str:<15}")
        
        k += 1
    
    print(f"\n  扫描结束: k={k}, 已选择={len(selections)}个文档")
    
    # 5. 检查是否需要Easy-Negative
    remaining = 9 - count_both
    need_en = remaining % 2
    
    print(f"\n🎨 Easy-Negative补漆分析:")
    print(f"  Both文档数: {count_both}")
    print(f"  剩余位置: 9 - {count_both} = {remaining}")
    print(f"  剩余位置奇偶: {'奇数' if remaining % 2 == 1 else '偶数'}")
    print(f"  是否需要补漆: {'✓ 是' if need_en else '✗ 否'}")
    
    if need_en and len(selections) == 9:
        print(f"  ⚠️ 需要补漆但已有9个文档，需要移除1个")
        if count_a_only > count_b_only:
            print(f"  → 移除1个A-Only (因为A:{count_a_only} > B:{count_b_only})")
        elif count_b_only > count_a_only:
            print(f"  → 移除1个B-Only (因为B:{count_b_only} > A:{count_a_only})")
        else:
            print(f"  → A和B相等，移除最后添加的那个")
    
    # 6. 执行实际的交错
    print(f"\n✨ 执行实际交错:")
    result = rank_based_interleaving(
        df_A_query, df_B_query,
        {qid: easy_negatives.get(qid, []) if isinstance(easy_negatives, dict) else easy_negatives},
        balance_threshold=balance_threshold,
        smart_easy_negative=True
    )
    
    print(f"\n📄 最终结果:")
    print(f"  {'排名':<6} {'docno':<25} {'来源':<15}")
    print(f"  {'-'*50}")
    for idx, row in result.iterrows():
        marker = "✓" if row['origin_label'] == 'Easy-Negative' else "  "
        print(f"{marker} {row['rank']:<6} {row['docno']:<25} {row['origin_label']:<15}")
    
    # 7. 统计分析
    stats = analyze_interleaving_results(result)
    row = stats[stats['qid'] == qid].iloc[0]
    
    print(f"\n📊 统计结果:")
    print(f"  Both: {row['Both']}")
    print(f"  A-Only: {row['A-Only']}")
    print(f"  B-Only: {row['B-Only']}")
    print(f"  Easy-Negative: {row['Easy-Negative']}")
    print(f"  总计: {row['total_docs']}")
    
    # 8. 分析问题
    print(f"\n🔍 问题分析:")
    if row['A-Only'] != row['B-Only']:
        diff = abs(row['A-Only'] - row['B-Only'])
        more = 'A' if row['A-Only'] > row['B-Only'] else 'B'
        print(f"  ⚠️ 不平衡检测: {more}-Only比另一方多 {diff} 个")
        
        if diff <= balance_threshold:
            print(f"  ✓ 在允许的阈值内 (threshold={balance_threshold})")
        else:
            print(f"  ❌ 超出阈值！可能的原因:")
            print(f"    1. 扫描深度不够")
            print(f"    2. 某一方文档不足")
            print(f"    3. 平衡逻辑触发时机问题")
    else:
        print(f"  ✓ A和B完美平衡!")
    
    if row['Easy-Negative'] != need_en:
        print(f"  ⚠️ Easy-Negative数量异常:")
        print(f"    预期: {need_en} 个")
        print(f"    实际: {row['Easy-Negative']} 个")
    
    print(f"\n" + "=" * 100)
    
    return result

In [ ]:
result = diagnose_query(
    df_A=df_colbert,
    df_B=df_colbert_prf,
    qid='1121402',
    easy_negatives=dataset.get_qrels(),
    balance_threshold=2
)